# Lab 10 — Vertex AI Pipelines: Orchestrating End-to-End Workflow

**Dataset:** Palmer Penguins (`bigquery-public-data.ml_datasets.penguins`)  
**Task:** Build a Kubeflow pipeline: data prep → train → evaluate → conditional deploy  
**Container Strategy:** `python:3.10` base with pinned sklearn for components; matching sklearn serving image  
**Exam Relevance:** Very high — KFP components, `dsl.Condition`, pipeline parameters, scheduling

---

### What You'll Learn
1. KFP `@component` decorator — lightweight pipeline components
2. `dsl.Condition` — conditional branching (deploy vs skip)
3. Pipeline compilation, submission, and parameterization
4. `PipelineJobSchedule` — recurring pipeline execution

### Parts
| Part | Focus |
|------|-------|
| 1 | Setup + Data Exploration |
| 2 | Pipeline Components (Lightweight) |
| 3 | Pipeline Definition + Execution |
| 4 | Scheduling + Cleanup |

### Container Strategy Decision

**Critical lesson from Labs 2, 7, 8, 9:** sklearn/numpy version mismatches between training
and serving containers cause binary incompatibility errors. The sklearn prediction images
(`sklearn-cpu.1-0`, `sklearn-cpu.1-3`) use old Python/pip versions that can't install modern
KFP dependencies, so they can't be used as component base images.

**Solution:** Use `python:3.10` as the base image for all components (modern pip, KFP-compatible),
and pin `scikit-learn` to match the serving container version exactly.

```
Component base image: python:3.10  + scikit-learn==1.3.2  (via packages_to_install)
Serving container:    us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                      Same sklearn version = no serialization mismatch
```

### Estimated Cost: $2–4

---
## Part 1: Setup + Data Exploration

### 1.1 Install Dependencies

We need `kfp` (Kubeflow Pipelines SDK) for pipeline definition and `google-cloud-pipeline-components`
for pre-built GCP-integrated components. The KFP SDK lets us define pipelines as Python functions
and compile them to a JSON spec that Vertex AI can execute.

In [1]:
# Install pipeline SDKs
# kfp: Kubeflow Pipelines SDK for defining components and pipelines
# google-cloud-pipeline-components: pre-built components for GCP services
!pip install --quiet kfp google-cloud-pipeline-components google-cloud-aiplatform google-cloud-bigquery db-dtypes


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### 1.2 Configuration

Set project constants. Same pattern as previous labs — everything defined once at the top.

In [2]:
import os
from datetime import datetime

# --- Project Configuration ---
PROJECT_ID = "carty-470812"
REGION = "us-central1"
BUCKET_NAME = f"{PROJECT_ID}"
BUCKET_URI = f"gs://{BUCKET_NAME}"
PIPELINE_ROOT = f"{BUCKET_URI}/lab10-pipelines"

# Serving container — sklearn 1.3 prediction image
SKLEARN_SERVING_IMAGE = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"

# Component dependencies — pinned to match the serving container
COMPONENT_DEPS = ["scikit-learn==1.3.2", "pandas", "joblib"]

# Timestamp for unique resource naming (hyphens, not underscores — Vertex AI requirement)
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Project:        {PROJECT_ID}")
print(f"Region:         {REGION}")
print(f"Pipeline Root:  {PIPELINE_ROOT}")
print(f"Serving Image:  {SKLEARN_SERVING_IMAGE}")
print(f"Component Deps: {COMPONENT_DEPS}")
print(f"Timestamp:      {TIMESTAMP}")

Project:        carty-470812
Region:         us-central1
Pipeline Root:  gs://carty-470812/lab10-pipelines
Serving Image:  us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest
Component Deps: ['scikit-learn==1.3.2', 'pandas', 'joblib']
Timestamp:      20260308-102145


### 1.3 Initialize Clients

In [3]:
from google.cloud import bigquery, aiplatform

bq_client = bigquery.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

print("Clients initialized.")

Clients initialized.


### 1.4 Explore the Penguins Dataset

Palmer Penguins is a clean, small dataset with 3 species classes. Let's see what we're working with.
This is a quick EDA — the focus of this lab is the pipeline, not the model.

In [4]:
# Query the full dataset
query = """
SELECT *
FROM `bigquery-public-data.ml_datasets.penguins`
"""

df = bq_client.query(query).to_dataframe()
print(f"Shape: {df.shape}")
print(f"\nSpecies distribution:")
print(df['species'].value_counts())
print(f"\nNull counts:")
print(df.isnull().sum())
print(f"\nSample rows:")
df.head()

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Shape: (344, 7)

Species distribution:
species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

Null counts:
species               0
island                0
culmen_length_mm      2
culmen_depth_mm       2
flipper_length_mm     2
body_mass_g           2
sex                  10
dtype: int64

Sample rows:


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie Penguin (Pygoscelis adeliae),Dream,36.6,18.4,184.0,3475.0,FEMALE
1,Adelie Penguin (Pygoscelis adeliae),Dream,39.8,19.1,184.0,4650.0,MALE
2,Adelie Penguin (Pygoscelis adeliae),Dream,40.9,18.9,184.0,3900.0,MALE
3,Chinstrap penguin (Pygoscelis antarctica),Dream,46.5,17.9,192.0,3500.0,FEMALE
4,Adelie Penguin (Pygoscelis adeliae),Dream,37.3,16.8,192.0,3000.0,FEMALE


In [5]:
# Check feature types and basic stats
df.describe()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,200.915205,4201.754386
std,5.459584,1.974793,14.061714,801.954536
min,32.100000,13.100000,172.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.000000,4750.000000
max,59.600000,21.500000,231.000000,6300.000000


### 1.5 Data Quality Notes

The penguins dataset has ~344 rows and ~11 rows with nulls (mostly in `sex` and measurement columns).
For this lab, we'll simply drop nulls — the focus is pipeline mechanics, not data imputation.

**Features we'll use:**
- `island` (categorical — Biscoe, Dream, Torgersen)
- `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g` (numeric)
- `sex` (categorical — MALE, FEMALE)

**Target:** `species` (Adelie, Chinstrap, Gentoo)

> **Exam note:** In a production pipeline, you'd handle nulls with imputation strategies.
> The pipeline pattern stays the same — you'd just add more logic to the `data_prep` component.

---
## Part 2: Pipeline Components (Lightweight)

### How Lightweight Components Work

With KFP's `@component` decorator, each pipeline step is a self-contained Python function.
KFP handles:
- **Containerization:** Builds a container image from the base image + installed packages
- **Serialization:** Inputs/outputs are serialized to/from GCS automatically
- **Isolation:** Each component runs in its own container — no shared state

This is fundamentally different from our previous labs where we wrote training scripts and
submitted them as `CustomTrainingJob`s. Here, the pipeline framework manages the containers.

```
Lightweight @component:         Heavyweight (Lab 2-9 pattern):
┌───────────────────────┐         ┌─────────────────────┐
│ Python function       │         │ training_script.py  │
│ + base_image          │  vs.    │ + Dockerfile        │
│ + packages_to_install │         │ + CustomTrainingJob │
│ = KFP builds it       │         │ = You build it      │
└───────────────────────┘         └─────────────────────┘
```

### 2.1 Imports for Component Definition

We import the KFP SDK for defining components and the DSL for pipeline logic.

In [6]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, component

### 2.2 Component 1: Data Preparation

This component queries BigQuery, cleans the data, encodes categoricals, splits train/test,
and writes CSVs to GCS.

**Key KFP concepts here:**
- `@component` decorator with `base_image` and `packages_to_install`
- `Output[Dataset]` — KFP artifact type. KFP manages the GCS path; we just write to `artifact.path`
- All imports must be **inside** the function (each component is an isolated container)

**Container note:** We use `python:3.10` as the base and install our pinned sklearn + pandas
plus the BigQuery client libraries.

In [7]:
@component(
    base_image="python:3.10",
    packages_to_install=COMPONENT_DEPS + ["google-cloud-bigquery", "db-dtypes", "pyarrow"],
)
def data_prep(
    project_id: str,
    train_dataset: Output[Dataset],
    test_dataset: Output[Dataset],
    test_size: float = 0.2,
    random_state: int = 42,
):
    """Query penguins from BigQuery, clean, encode, and split."""
    from google.cloud import bigquery
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder

    # Query BigQuery
    client = bigquery.Client(project=project_id)
    query = """
    SELECT
        island,
        culmen_length_mm,
        culmen_depth_mm,
        flipper_length_mm,
        body_mass_g,
        sex,
        species
    FROM `bigquery-public-data.ml_datasets.penguins`
    """
    df = client.query(query).to_dataframe()
    print(f"Queried {len(df)} rows from BigQuery")

    # Drop nulls
    df = df.dropna()
    print(f"After dropping nulls: {len(df)} rows")

    # Encode categoricals
    for col in ["island", "sex"]:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        print(f"Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

    # Encode target
    le_species = LabelEncoder()
    df["species"] = le_species.fit_transform(df["species"])
    print(f"Encoded species: {dict(zip(le_species.classes_, le_species.transform(le_species.classes_)))}")

    # Split
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df["species"]
    )
    print(f"Train: {len(train_df)}, Test: {len(test_df)}")

    # Write to KFP-managed artifact paths
    train_df.to_csv(train_dataset.path, index=False)
    test_df.to_csv(test_dataset.path, index=False)
    print(f"Wrote train to {train_dataset.path}")
    print(f"Wrote test to {test_dataset.path}")

### 2.3 Component 2: Train Model

Trains a RandomForestClassifier and serializes it with joblib.

**Key KFP concepts:**
- `Output[Model]` — KFP artifact type for models. KFP tracks this in pipeline lineage automatically.
- We save as `model.joblib` inside a directory at `model_artifact.path`. This directory structure
  matches what the sklearn serving container expects when we upload to Model Registry.
- `packages_to_install` uses `COMPONENT_DEPS` to pin sklearn to 1.3.2, matching the serving container.

In [8]:
@component(
    base_image="python:3.10",
    packages_to_install=COMPONENT_DEPS,
)
def train_model(
    train_dataset: Input[Dataset],
    model_artifact: Output[Model],
    n_estimators: int = 100,
    max_depth: int = 5,
    random_state: int = 42,
):
    """Train a RandomForestClassifier on the training data."""
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    import joblib
    import os

    # Load training data
    train_df = pd.read_csv(train_dataset.path)
    X_train = train_df.drop("species", axis=1)
    y_train = train_df["species"]
    print(f"Training on {len(X_train)} samples, {X_train.shape[1]} features")

    # Train
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )
    model.fit(X_train, y_train)

    train_accuracy = model.score(X_train, y_train)
    print(f"Train accuracy: {train_accuracy:.4f}")

    # Save model into artifact directory as model.joblib
    # The sklearn serving container expects model.joblib at the root of the artifact URI
    os.makedirs(model_artifact.path, exist_ok=True)
    model_path = os.path.join(model_artifact.path, "model.joblib")
    joblib.dump(model, model_path)
    model_artifact.metadata["framework"] = "sklearn"
    model_artifact.metadata["n_estimators"] = n_estimators
    model_artifact.metadata["max_depth"] = max_depth
    model_artifact.metadata["train_accuracy"] = float(train_accuracy)
    print(f"Model saved to {model_path}")

### 2.4 Component 3: Evaluate Model

Loads the trained model and test data, computes metrics, and returns accuracy as a float output.

**Key KFP concept:** Returning a simple Python type (`float`) as a component output.
KFP serializes this automatically. The accuracy value will be used in `dsl.Condition` to
decide whether to deploy.

**Note:** When a component has multiple outputs (like `metrics` + the return value), you must
reference the return value by name (`eval_task.outputs["Output"]`) rather than `eval_task.output`.

In [9]:
@component(
    base_image="python:3.10",
    packages_to_install=COMPONENT_DEPS,
)
def evaluate_model(
    test_dataset: Input[Dataset],
    model_artifact: Input[Model],
    metrics: Output[Metrics],
) -> float:
    """Evaluate the model on the test set. Returns accuracy."""
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    import joblib
    import os

    # Load test data and model
    test_df = pd.read_csv(test_dataset.path)
    X_test = test_df.drop("species", axis=1)
    y_test = test_df["species"]

    model_path = os.path.join(model_artifact.path, "model.joblib")
    model = joblib.load(model_path)
    print(f"Evaluating on {len(X_test)} samples")

    # Predict and compute metrics
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test F1 (weighted): {f1:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

    # Log metrics to KFP (visible in pipeline UI)
    metrics.log_metric("accuracy", accuracy)
    metrics.log_metric("f1_weighted", f1)
    metrics.log_metric("test_samples", len(X_test))

    return accuracy

### 2.5 Component 4: Deploy Model

Uploads the model to Vertex AI Model Registry and deploys to an endpoint.

**Key KFP concept:** This component only runs if the accuracy condition is met.
It uses the Vertex AI SDK inside the component to handle deployment.

**Container note:** This component uses the AI Platform SDK, not sklearn directly, so we use
`python:3.10` as the base and install `google-cloud-aiplatform`. The serving container
(specified in `Model.upload`) is `SKLEARN_SERVING_IMAGE`.

In [10]:
@component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform"],
)
def deploy_model(
    model_artifact: Input[Model],
    project_id: str,
    region: str,
    timestamp: str,
    serving_image: str,
) -> str:
    """Upload model to Vertex AI Model Registry and deploy to endpoint."""
    from google.cloud import aiplatform

    aiplatform.init(project=project_id, location=region)

    # model_artifact.uri points to the directory containing model.joblib
    vertex_model = aiplatform.Model.upload(
        display_name=f"penguins-rf-{timestamp}",
        artifact_uri=model_artifact.uri,
        serving_container_image_uri=serving_image,
    )
    print(f"Model uploaded: {vertex_model.resource_name}")

    # Create endpoint and deploy
    endpoint = aiplatform.Endpoint.create(
        display_name=f"penguins-endpoint-{timestamp}",
    )
    print(f"Endpoint created: {endpoint.resource_name}")

    vertex_model.deploy(
        endpoint=endpoint,
        deployed_model_display_name=f"penguins-deployed-{timestamp}",
        machine_type="n1-standard-2",
        min_replica_count=1,
        max_replica_count=1,
    )
    print(f"Model deployed to endpoint: {endpoint.resource_name}")

    return endpoint.resource_name

### 2.6 Component Summary

We now have 4 lightweight components:

| Component | Base Image | packages_to_install | Outputs |
|-----------|-----------|---------------------|---------|
| `data_prep` | python:3.10 | COMPONENT_DEPS + BQ libs | Output[Dataset] x2 |
| `train_model` | python:3.10 | COMPONENT_DEPS | Output[Model] |
| `evaluate_model` | python:3.10 | COMPONENT_DEPS | Output[Metrics], float |
| `deploy_model` | python:3.10 | google-cloud-aiplatform | endpoint name (str) |

**How data flows between components:**
```
data_prep ──train_dataset──► train_model ──model_artifact──┐
    │                                                       │
    └───test_dataset──► evaluate_model ◄────────────────────┘
                              │
                          accuracy (float)
                              │
                        dsl.Condition
                         (accuracy > threshold?)
                              │
                    ┌─────────┴─────────┐
                    ▼                   ▼
              deploy_model           (skip)
```

> **Exam note:** KFP serializes all data between components via GCS. There is no shared memory
> or filesystem between steps. This is why imports must be inside each function — each runs
> in its own container.

---
## Part 3: Pipeline Definition + Execution

Now we wire the components together into a pipeline with conditional deployment logic.

### 3.1 Define the Pipeline

The `@dsl.pipeline` decorator defines the pipeline graph. Inside, we call components
like regular functions — KFP interprets these as pipeline steps and wires the DAG based
on input/output dependencies.

**Key KFP concept:** `dsl.Condition` creates a conditional branch. The step inside the
`with` block only executes if the condition is true at runtime.

**Note:** `dsl.Condition` is deprecated in favor of `dsl.If`, but both work. The exam
may reference either.

In [11]:
@dsl.pipeline(
    name="penguins-classification-pipeline",
    description="Train, evaluate, and conditionally deploy a penguin species classifier.",
    pipeline_root=PIPELINE_ROOT,
)
def penguins_pipeline(
    project_id: str = PROJECT_ID,
    region: str = REGION,
    timestamp: str = TIMESTAMP,
    serving_image: str = SKLEARN_SERVING_IMAGE,
    accuracy_threshold: float = 0.90,
    n_estimators: int = 100,
    max_depth: int = 5,
):
    """Pipeline: data prep -> train -> evaluate -> conditional deploy."""

    # Step 1: Data preparation
    data_prep_task = data_prep(project_id=project_id)

    # Step 2: Train model
    train_task = train_model(
        train_dataset=data_prep_task.outputs["train_dataset"],
        n_estimators=n_estimators,
        max_depth=max_depth,
    )

    # Step 3: Evaluate model
    eval_task = evaluate_model(
        test_dataset=data_prep_task.outputs["test_dataset"],
        model_artifact=train_task.outputs["model_artifact"],
    )

    # Step 4: Conditional deployment
    # Use eval_task.outputs["Output"] (not eval_task.output) because evaluate_model
    # has multiple outputs (metrics + return value)
    with dsl.Condition(
        eval_task.outputs["Output"] >= accuracy_threshold,
        name="deploy-if-accurate",
    ):
        deploy_task = deploy_model(
            model_artifact=train_task.outputs["model_artifact"],
            project_id=project_id,
            region=region,
            timestamp=timestamp,
            serving_image=serving_image,
        )

/var/folders/gm/vjgn8n7j4jq6v03c3nq66t8m0000gn/T/ipykernel_9108/2084687594.py:36: DeprecationWarning: dsl.Condition is deprecated. Please use dsl.If instead.
  with dsl.Condition(


### 3.2 Compile the Pipeline

KFP compiles the Python pipeline definition to a JSON specification that Vertex AI
can execute. This JSON describes the DAG, container specs, and artifact dependencies.

In [12]:
from kfp import compiler

PIPELINE_JSON = "penguins_pipeline.json"
compiler.Compiler().compile(
    pipeline_func=penguins_pipeline,
    package_path=PIPELINE_JSON,
)
print(f"Pipeline compiled to {PIPELINE_JSON}")
print(f"File size: {os.path.getsize(PIPELINE_JSON) / 1024:.1f} KB")

Pipeline compiled to penguins_pipeline.json
File size: 21.3 KB


### 3.3 Run 1 — Realistic Threshold (Should Deploy)

We set `accuracy_threshold=0.90`. RandomForest on penguins should easily exceed this
(expect ~95-98% accuracy), so the deploy branch should execute.

In [13]:
from google.cloud.aiplatform import PipelineJob

# Run 1: Threshold the model should pass
job_pass = PipelineJob(
    display_name=f"penguins-pipeline-pass-{TIMESTAMP}",
    template_path=PIPELINE_JSON,
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "project_id": PROJECT_ID,
        "region": REGION,
        "timestamp": TIMESTAMP,
        "serving_image": SKLEARN_SERVING_IMAGE,
        "accuracy_threshold": 0.90,
        "n_estimators": 100,
        "max_depth": 5,
    },
)

job_pass.submit()
print(f"Pipeline job submitted: {job_pass.display_name}")
print(f"Job resource name: {job_pass.resource_name}")
print(f"\nView in console:")
print(f"https://console.cloud.google.com/vertex-ai/pipelines/runs?project={PROJECT_ID}")

Creating PipelineJob
PipelineJob created. Resource name: projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308102317
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308102317')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/penguins-classification-pipeline-20260308102317?project=873708835509
Pipeline job submitted: penguins-pipeline-pass-20260308-102145
Job resource name: projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308102317

View in console:
https://console.cloud.google.com/vertex-ai/pipelines/runs?project=carty-470812


In [14]:
# Wait for completion (this will take a few minutes)
# The deploy step takes the longest due to endpoint creation
job_pass.wait()
print(f"\nPipeline state: {job_pass.state}")

PipelineJob run completed. Resource name: projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308102317

Pipeline state: 4


### 3.4 Inspect Run 1 Results

Let's verify the pipeline executed correctly and the deploy branch was triggered.

In [16]:
# The pipeline graph in the Vertex AI console is the best way to visualize this.
print("View the pipeline graph in the Vertex AI console:")
print(f"https://console.cloud.google.com/vertex-ai/pipelines/runs?project={PROJECT_ID}")
print("\nExpected result:")
print("  data-prep          -> green (completed)")
print("  train-model        -> green (completed)")
print("  evaluate-model     -> green (completed)")
print("  deploy-if-accurate -> green (condition met, deploy executed)")

View the pipeline graph in the Vertex AI console:
https://console.cloud.google.com/vertex-ai/pipelines/runs?project=carty-470812

Expected result:
  data-prep          -> green (completed)
  train-model        -> green (completed)
  evaluate-model     -> green (completed)
  deploy-if-accurate -> green (condition met, deploy executed)


### 3.5 Test the Deployed Endpoint

Let's verify the endpoint actually works by sending a prediction request.

In [17]:
# Find the deployed endpoint
endpoints = [
    e for e in aiplatform.Endpoint.list()
    if "penguins-endpoint" in e.display_name
]

if endpoints:
    endpoint = endpoints[0]
    print(f"Found endpoint: {endpoint.display_name}")

    # Sample prediction: a penguin with specific measurements
    # Features: island, culmen_length_mm, culmen_depth_mm, flipper_length_mm, body_mass_g, sex
    test_instance = [0, 39.1, 18.7, 181.0, 3750.0, 1]  # Encoded values

    prediction = endpoint.predict(instances=[test_instance])
    print(f"\nInput:      {test_instance}")
    print(f"Prediction: {prediction.predictions}")
    print(f"(0=Adelie, 1=Chinstrap, 2=Gentoo)")
else:
    print("No endpoint found — check pipeline logs.")

Found endpoint: penguins-endpoint-20260308-102145

Input:      [0, 39.1, 18.7, 181.0, 3750.0, 1]
Prediction: [0.0]
(0=Adelie, 1=Chinstrap, 2=Gentoo)


### 3.6 Run 2 — Impossible Threshold (Should Skip Deploy)

Now we set `accuracy_threshold=1.0` — no model will hit 100% accuracy, so the deploy
branch should be skipped. This verifies our `dsl.Condition` logic works in both directions.

In [18]:
TIMESTAMP_2 = datetime.now().strftime("%Y%m%d-%H%M%S")

job_skip = PipelineJob(
    display_name=f"penguins-pipeline-skip-{TIMESTAMP_2}",
    template_path=PIPELINE_JSON,
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "project_id": PROJECT_ID,
        "region": REGION,
        "timestamp": TIMESTAMP_2,
        "serving_image": SKLEARN_SERVING_IMAGE,
        "accuracy_threshold": 1.0,  # Impossible — deploy should be skipped
        "n_estimators": 100,
        "max_depth": 5,
    },
)

job_skip.submit()
print(f"Pipeline job submitted: {job_skip.display_name}")
print("This run should skip the deploy step.")

Creating PipelineJob
PipelineJob created. Resource name: projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308105141
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308105141')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/penguins-classification-pipeline-20260308105141?project=873708835509
Pipeline job submitted: penguins-pipeline-skip-20260308-105141
This run should skip the deploy step.


In [19]:
job_skip.wait()
print(f"Pipeline state: {job_skip.state}")
print("\nCheck the pipeline graph. The deploy step should show as SKIPPED.")
print(f"https://console.cloud.google.com/vertex-ai/pipelines/runs?project={PROJECT_ID}")

PipelineJob run completed. Resource name: projects/873708835509/locations/us-central1/pipelineJobs/penguins-classification-pipeline-20260308105141
Pipeline state: 4

Check the pipeline graph. The deploy step should show as SKIPPED.
https://console.cloud.google.com/vertex-ai/pipelines/runs?project=carty-470812


### 3.7 Part 3 Summary

**What we demonstrated:**
- **Pipeline parameterization:** `accuracy_threshold`, `n_estimators`, `max_depth` are all configurable at runtime
- **Conditional branching:** `dsl.Condition` only deploys when the model meets our quality bar
- **Two runs:** Verified both branches (deploy and skip) work correctly
- **End-to-end serving:** Prediction request against the deployed endpoint

> **Exam note:** `dsl.Condition` is the primary mechanism for conditional logic in KFP pipelines.
> Common patterns include: deploy-if-accurate (what we did), retrain-if-drift-detected, and
> skip-expensive-steps-for-dev-runs. The exam loves questions about how to implement
> "only deploy if accuracy exceeds X" — this is exactly that pattern.

---
## Part 4: Scheduling + Cleanup

### Why Schedule Pipelines?

In production, you don't run pipelines manually. Common patterns:
- **Daily retraining:** Fresh data arrives nightly → retrain and evaluate each morning
- **Weekly evaluation:** Check for drift, re-evaluate model quality weekly
- **Event-triggered:** New data lands in BigQuery → trigger pipeline (via Cloud Functions/Pub/Sub → pipeline)

Vertex AI supports cron-based scheduling natively via `PipelineJobSchedule`.

### 4.1 Create a Pipeline Schedule

We'll create a schedule that would run daily at 6 AM UTC, then immediately pause and
delete it to avoid costs.

> **Exam note:** `PipelineJobSchedule` uses standard cron syntax. The exam may ask about
> scheduling patterns — know that Vertex AI supports both cron schedules and API-triggered runs.

In [20]:
# Create a scheduled pipeline run
# We create the schedule, verify it exists, then immediately clean it up

schedule_job = PipelineJob(
    display_name=f"penguins-scheduled-{TIMESTAMP}",
    template_path=PIPELINE_JSON,
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "project_id": PROJECT_ID,
        "region": REGION,
        "timestamp": TIMESTAMP,
        "serving_image": SKLEARN_SERVING_IMAGE,
        "accuracy_threshold": 0.90,
        "n_estimators": 100,
        "max_depth": 5,
    },
)

schedule = schedule_job.create_schedule(
    display_name=f"penguins-schedule-{TIMESTAMP}",
    cron="0 6 * * *",  # Daily at 6 AM UTC (minute hour day-of-month month day-of-week)
    max_concurrent_run_count=1,
    max_run_count=10,  # Safety limit: stop after 10 runs
)

print(f"Schedule created: {schedule.display_name}")
print(f"Cron: 0 6 * * * (daily at 6 AM UTC)")
print(f"Max runs: 10")
print(f"\nView schedules:")
print(f"https://console.cloud.google.com/vertex-ai/pipelines/schedules?project={PROJECT_ID}")

Creating PipelineJobSchedule
PipelineJobSchedule created. Resource name: projects/873708835509/locations/us-central1/schedules/6420580558211907584
To use this PipelineJobSchedule in another session:
schedule = aiplatform.PipelineJobSchedule.get('projects/873708835509/locations/us-central1/schedules/6420580558211907584')
View Schedule:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/schedules/6420580558211907584?project=873708835509
Schedule created: penguins-schedule-20260308-102145
Cron: 0 6 * * * (daily at 6 AM UTC)
Max runs: 10

View schedules:
https://console.cloud.google.com/vertex-ai/pipelines/schedules?project=carty-470812


### 4.2 Manage the Schedule

In production, you'd pause schedules during maintenance windows and resume them after.

In [21]:
# Pause the schedule
schedule.pause()
print(f"Schedule paused: {schedule.display_name}")

# In production, you'd resume with:
# schedule.resume()

Schedule paused: penguins-schedule-20260308-102145


In [22]:
# Delete the schedule (avoid any accidental runs and costs)
schedule.delete()
print(f"Schedule deleted: {schedule.display_name}")
print("\nSchedule lifecycle complete: create -> pause -> delete")

Deleting PipelineJobSchedule : projects/873708835509/locations/us-central1/schedules/6420580558211907584
PipelineJobSchedule deleted. . Resource name: projects/873708835509/locations/us-central1/schedules/6420580558211907584
Deleting PipelineJobSchedule resource: projects/873708835509/locations/us-central1/schedules/6420580558211907584
Delete PipelineJobSchedule backing LRO: projects/873708835509/locations/us-central1/operations/2946873002765058048
PipelineJobSchedule resource projects/873708835509/locations/us-central1/schedules/6420580558211907584 deleted.
Schedule deleted: penguins-schedule-20260308-102145

Schedule lifecycle complete: create -> pause -> delete


### 4.3 Scheduling Patterns for the Exam

| Pattern | Mechanism | When to Use |
|---------|-----------|-------------|
| **Cron schedule** | `PipelineJobSchedule` | Regular intervals (daily, weekly) |
| **Event-triggered** | Cloud Function → PipelineJob.submit() | New data arrival, drift alert |
| **Manual** | Console or SDK `PipelineJob.submit()` | Ad-hoc retraining, debugging |
| **CI/CD triggered** | Cloud Build → PipelineJob.submit() | Code changes to training script |

> **Exam tip:** If the question says "retrain automatically when new data arrives in BigQuery,"
> the answer is NOT a cron schedule — it's an event-triggered pattern using Cloud Functions or
> Eventarc listening for BigQuery insert events, which then submits a pipeline run.

### 4.4 Resource Cleanup

Clean up all resources created during this lab.

In [ ]:
from google.cloud import storage

print("=" * 60)
print("RESOURCE CLEANUP")
print("=" * 60)

# 1. Delete endpoints (must undeploy models first)
print("\n--- Endpoints ---")
endpoints = [
    e for e in aiplatform.Endpoint.list()
    if "penguins-endpoint" in e.display_name
]
for endpoint in endpoints:
    print(f"Undeploying and deleting: {endpoint.display_name}")
    try:
        endpoint.undeploy_all()
        endpoint.delete()
        print(f"  Deleted")
    except Exception as e:
        print(f"  Error: {e}")

# 2. Delete models from Model Registry
print("\n--- Models ---")
models = [
    m for m in aiplatform.Model.list()
    if "penguins" in m.display_name
]
for model in models:
    print(f"Deleting model: {model.display_name}")
    try:
        model.delete()
        print(f"  Deleted")
    except Exception as e:
        print(f"  Error: {e}")

# 3. Delete pipeline runs
print("\n--- Pipeline Jobs ---")
pipeline_jobs = [
    j for j in aiplatform.PipelineJob.list()
    if "penguins" in j.display_name
]
for job in pipeline_jobs:
    print(f"Deleting pipeline job: {job.display_name}")
    try:
        job.delete()
        print(f"  Deleted")
    except Exception as e:
        print(f"  Error: {e}")

# 4. Delete any remaining schedules
print("\n--- Schedules ---")
from google.cloud.aiplatform import PipelineJobSchedule
schedules = [
    s for s in PipelineJobSchedule.list()
    if "penguins" in s.display_name
]
for s in schedules:
    print(f"Deleting schedule: {s.display_name}")
    try:
        s.delete()
        print(f"  Deleted")
    except Exception as e:
        print(f"  Error: {e}")

# 5. Clean up GCS artifacts
print("\n--- GCS Cleanup ---")
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
blobs = list(bucket.list_blobs(prefix="lab10-pipelines/"))
if blobs:
    print(f"Deleting {len(blobs)} objects under gs://{BUCKET_NAME}/lab10-pipelines/")
    for blob in blobs:
        blob.delete()
    print("  GCS artifacts deleted")
else:
    print("No GCS artifacts found.")

print("\n" + "=" * 60)
print("CLEANUP COMPLETE")
print("=" * 60)

### 4.5 Local File Cleanup

In [ ]:
import os

for f in ["penguins_pipeline.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

print("\nLocal cleanup complete.")

---
## Lab 10 Summary

### What We Built
A Kubeflow Pipeline on Vertex AI that:
1. **Queries and prepares** penguin data from BigQuery
2. **Trains** a RandomForest classifier using lightweight KFP components
3. **Evaluates** accuracy and logs metrics
4. **Conditionally deploys** to an endpoint only if accuracy exceeds a threshold
5. **Schedules** recurring execution with cron syntax

### Key Concepts for the Exam

| Concept | What to Know |
|---------|-------------|
| `@component` decorator | Defines lightweight pipeline steps; KFP builds the container |
| `dsl.Condition` | Conditional branching — deploy only if quality threshold is met |
| Pipeline parameters | Runtime-configurable values (threshold, hyperparameters) |
| KFP artifact types | `Dataset`, `Model`, `Metrics` — serialized via GCS between steps |
| Lightweight vs heavyweight | In-component training vs launching a CustomJob |
| `PipelineJobSchedule` | Cron-based recurring pipeline execution |
| Event-triggered pipelines | Cloud Functions / Eventarc → PipelineJob.submit() |

### When to Use Lightweight vs Heavyweight

| Factor | Lightweight `@component` | Heavyweight `CustomJob` |
|--------|------------------------|------------------------|
| Training time | Minutes | Hours |
| Data size | < 1 GB | Any size |
| GPU required | No | Yes |
| Custom Docker image | No (KFP builds it) | Yes (you control it) |
| HP Tuning integration | Manual only | Vertex AI HP Tuning |
| Typical use case | Sklearn on small data | TF/PyTorch on large data |

> **Exam pattern:** "A team trains a large TensorFlow model requiring GPUs. They want to
> orchestrate data prep, training, evaluation, and conditional deployment. What should they use?"
> Answer: Vertex AI Pipeline with a component that launches a CustomContainerTrainingJob
> (heavyweight pattern). The pipeline component itself doesn't need GPUs — it just submits
> the training job and waits.

### Key Learnings
- **Lightweight components are surprisingly powerful:** For small models like sklearn on tabular data, there's no need for a separate training VM
- **`dsl.Condition` is the exam's favorite:** Any "deploy only if X" question → `dsl.Condition`
- **Imports must be inside components:** Each runs in an isolated container with no shared state
- **Pipeline parameters make pipelines reusable:** Same pipeline, different thresholds/hyperparameters
- **Scheduling ≠ event-triggered:** Know the difference for the exam
- **Container consistency (again!):** Pin sklearn versions in training components to match the serving container — recurring theme across Labs 2, 7, 8, 9, and now 10
- **Prediction images ≠ component base images:** Google's sklearn prediction containers have old Python/pip that can't install KFP; use `python:3.10` + pinned deps instead
- **Multi-output components:** When a component has multiple outputs, reference by name (`outputs["Output"]`) not `.output`

### GCP Services Used
- Vertex AI Pipelines (KFP)
- BigQuery (data source)
- Cloud Storage (pipeline artifacts)
- Vertex AI Model Registry
- Vertex AI Endpoints

---
### Appendix: Problems Encountered & Fixes

| # | Problem | Root Cause | Fix |
|---|---------|-----------|-----|
| 1 | deploy_model failed: "Model server never became ready" | Model saved as flat file (`model_artifact.joblib`) not in directory structure | Save as `model_artifact.path/model.joblib` (directory); use `model_artifact.uri` as `artifact_uri` |
| 2 | numpy.dtype binary incompatibility (Expected 96, got 88) | Training component used `sklearn-cpu.1-0` image but model evaluated/served by sklearn 1.3 | Use same sklearn version everywhere — pin `scikit-learn==1.3.2` in components, serve with `sklearn-cpu.1-3` |
| 3 | `kfp==2.16.0` not found in sklearn prediction image | sklearn prediction images have Python 3.7 / pip 20.1 — too old for modern KFP | Use `python:3.10` as component base image instead of prediction images |
| 4 | `ModuleNotFoundError: No module named 'pandas'` | sklearn prediction/training images don't include pandas | Add `"pandas"` to `packages_to_install` |
| 5 | `AttributeError: The task has multiple outputs` | `evaluate_model` returns float + has `Output[Metrics]`; `.output` is ambiguous | Use `eval_task.outputs["Output"]` to reference the return value |
| 6 | `dsl.Condition is deprecated` warning | KFP v2 prefers `dsl.If` | Both work; `dsl.If` is the newer API |

### Appendix: Walkthrough Notes

*To be filled in during lab execution.*